In [ ]:
# %%
# 01a — Environment setup (must come before importing jax)
import os

os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [ ]:
# %%
# 01 — Paths, run policy, and switches
from pathlib import Path
import json
import os
import sys
import warnings

warnings.filterwarnings("ignore")

PREPROC_ROOT = Path.home() / "projects" / "allen_multistim_clean_v2"
INDEX_CSV = PREPROC_ROOT / "indexes" / "preprocessed_index.csv"

BROOT = Path.home() / "projects" / "allen_bio_cleanbreak_v1"
CACHE_DIR = BROOT / "cache"
TABLE_DIR = BROOT / "tables"
FIG_DIR = BROOT / "figures"
FIG_LABELED_DIR = FIG_DIR / "labeled"
FIG_UNLABELED_DIR = FIG_DIR / "unlabeled"
RESULTS_DIR = BROOT / "results"
LOG_DIR = BROOT / "logs"

for p in [BROOT, CACHE_DIR, TABLE_DIR, FIG_DIR, FIG_LABELED_DIR, FIG_UNLABELED_DIR, RESULTS_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CFG = {
    # reproducibility
    "SEED": 0,

    # compute
    "N_JOBS_BUILD": min(90, os.cpu_count() or 8),
    "N_JOBS_MAP": min(90, os.cpu_count() or 8),

    # cohort policy
    "AREA_ALLOWLIST": ["VISp", "VISl", "VISal", "VISpm", "VISam", "VISrl"],
    "VISP_LAYER_ALLOWLIST": ["L2/3", "L4", "L5", "L6"],
    "MIN_EXPERIMENTS_PER_LABEL": 4,
    # Area uses n_match=60; VISp-depth uses n_match=45 in the reported matched cohort.
    "TARGET_N_MATCH_AREA": 60,
    "TARGET_N_MATCH_VISP": 60,

    # geometry
    "EPS_SPD": 1e-6,
    "N_SUBSAMPLES": 100,
    "SPLIT_HALF_REPEATS_FULL": 8,
    "SPLIT_HALF_REPEATS_MATCHED": 2,

    # baselines
    "RUN_DECODER_BASELINES": True,
    "RUN_MAPPING_BASELINES": True,
    "MAP_N_SPLITS": 10,
    "RIDGE_ALPHAS": [0.1, 1.0, 10.0, 100.0],
    "PLS_N_COMPONENTS_MAX": 10,

    # state module
    "RUN_OPTIONAL_STATE_MODULE": True,
    "RUN_SPEED_STILL_MAX": 0.25,
    "RUN_SPEED_RUN_MIN": 1.0,
    "MIN_STATE_TRIALS_PER_CONDITION": 2,
    "MIN_STATE_TOTAL_TRIALS": 120,

    # later apparatus
    "RUN_OPTIONAL_GROUPED_FAMILY": True,
    "RUN_OPTIONAL_INCREMENTAL_NUISANCE": True,
    "RUN_OPTIONAL_NMATCH_SWEEP": True,
    "RUN_OPTIONAL_COUPLING": True,

    # stats
    "N_BOOT": 5000,
    "N_PERM": 5000,

    # cache policy
    "FORCE_REBUILD_CACHE": True,
    "FORCE_REBUILD_MAPPING": True,

    # export
    "FIG_DPI": 220,
    "NOTEBOOK_VERSION": "bio_cleanbreak_v1",
}

with open(LOG_DIR / "config.json", "w") as f:
    json.dump(CFG, f, indent=2)

print("Python:", sys.version)
print(json.dumps(CFG, indent=2))

In [ ]:
# %%
# 02 — Imports
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import matplotlib as mpl

from joblib import Parallel, delayed
from itertools import combinations

from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import pdist, squareform
from scipy.stats import pearsonr

from sklearn.covariance import LedoitWolf
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
# %%
import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp

from joblib import Parallel, delayed
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import RidgeCV
from tqdm.auto import tqdm
from contextlib import contextmanager

print("JAX devices:", jax.devices())
jax.config.update("jax_enable_x64", True)

In [ ]:
# %%
# 03 — Plot style
COLORS = {
    "black": "#222222",
    "gray": "#8D8D8D",
    "blue": "#4C78A8",
    "orange": "#F28E2B",
    "green": "#59A14F",
    "purple": "#7A68A6",
    "red": "#B55D60",
}

METHOD_COLORS = {
    "full_sras": COLORS["black"],
    "shape_sras": COLORS["blue"],
    "naive_full_sras": COLORS["gray"],
    "naive_shape_sras": COLORS["green"],
    "cka": COLORS["orange"],
    "rsa": COLORS["gray"],
    "decoder_profile_corr": COLORS["green"],
    "ridge_score": COLORS["gray"],
    "pls_score": COLORS["purple"],
}

mpl.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def format_ax(ax, labeled=True, xlabel=None, ylabel=None, title=None, baseline=None):
    ax.yaxis.grid(True, color="#d9d9d9", linewidth=0.6, alpha=0.7)
    ax.xaxis.grid(False)
    ax.set_axisbelow(True)
    if baseline is not None:
        ax.axhline(baseline, color="#999999", linestyle="--", linewidth=0.9)
    if labeled:
        if xlabel is not None:
            ax.set_xlabel(xlabel)
        if ylabel is not None:
            ax.set_ylabel(ylabel)
        if title is not None:
            ax.set_title(title)
    else:
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_title("")
        ax.set_xticklabels([])
        ax.set_yticklabels([])

def save_figure(fig, stem, labeled=True):
    out_dir = FIG_LABELED_DIR if labeled else FIG_UNLABELED_DIR
    fig.savefig(out_dir / f"{stem}.svg", dpi=CFG["FIG_DPI"], bbox_inches="tight")
    fig.savefig(out_dir / f"{stem}.pdf", dpi=CFG["FIG_DPI"], bbox_inches="tight")
    plt.close(fig)

In [ ]:
# %%
# 04 — Load preprocessing index and restrict to static-ready rows
index_df = pd.read_csv(INDEX_CSV)

static_df = index_df[index_df["has_static_condition_summary"].fillna(False)].copy()
static_df = static_df[static_df["is_excitatory"].fillna(False)].copy()

print("All rows:", len(index_df))
print("Static summary-ready excitatory rows:", len(static_df))

display(
    static_df.groupby("targeted_structure")
    .size()
    .rename("n")
    .reset_index()
    .sort_values("n", ascending=False)
)

In [ ]:
# %%
# 05 — Cohort construction with adaptive matched-count selection
def choose_n_match(df, target_n_match, min_experiments_per_label):
    candidates = list(range(target_n_match, 19, -5))
    chosen_n_match = None
    chosen_df = None

    for n_match in candidates:
        tmp = df[df["n_cells"] >= n_match].copy()
        counts = tmp.groupby("label").size()

        ok = True
        for label in sorted(df["label"].unique()):
            if counts.get(label, 0) < min_experiments_per_label:
                ok = False
                break

        if ok:
            chosen_n_match = n_match
            chosen_df = tmp.reset_index(drop=True)
            break

    if chosen_n_match is None:
        raise RuntimeError("No feasible matched count found.")
    return chosen_n_match, chosen_df

area_df = static_df[static_df["targeted_structure"].isin(CFG["AREA_ALLOWLIST"])].copy()
area_df["label"] = area_df["targeted_structure"]

visp_df = static_df[
    (static_df["targeted_structure"] == "VISp")
    & (static_df["visp_layer_label"].isin(CFG["VISP_LAYER_ALLOWLIST"]))
].copy()
visp_df["label"] = visp_df["visp_layer_label"]

N_MATCH_AREA, area_df = choose_n_match(
    area_df,
    CFG["TARGET_N_MATCH_AREA"],
    CFG["MIN_EXPERIMENTS_PER_LABEL"],
)

N_MATCH_VISP, visp_df = choose_n_match(
    visp_df,
    CFG["TARGET_N_MATCH_VISP"],
    CFG["MIN_EXPERIMENTS_PER_LABEL"],
)

print("Area cohort:", len(area_df), "N_MATCH =", N_MATCH_AREA)
display(area_df.groupby("label").size().rename("n").reset_index())

print("VISp layer cohort:", len(visp_df), "N_MATCH =", N_MATCH_VISP)
display(visp_df.groupby("label").size().rename("n").reset_index())

In [ ]:
# %%
# 06 — Static-summary loader
def load_static_bundle(path_h5):
    with h5py.File(path_h5, "r") as f:
        meta = dict(f["meta"].attrs)

        out = {
            "path_h5": str(path_h5),
            "id": int(meta["id"]),
            "targeted_structure": str(meta["targeted_structure"]),
            "imaging_depth": int(meta["imaging_depth"]),
            "cre_line": str(meta["cre_line"]),
            "donor_name": str(meta["donor_name"]),
            "visp_layer_label": str(meta["visp_layer_label"]),
            "n_cells": int(meta["n_cells"]),

            "baseline_trials": f["static_gratings_summary/trials/baseline_mean_dff"][:].astype(np.float64),
            "response_trials": f["static_gratings_summary/trials/response_mean_dff"][:].astype(np.float64),
            "delta_trials": f["static_gratings_summary/trials/delta_mean_dff"][:].astype(np.float64),
            "trial_running_speed": f["static_gratings_summary/trials/running_speed_mean_cm_per_s"][:].astype(np.float64),

            "ori_vals": f["static_gratings_summary/design/orientation_values_deg"][:].astype(np.float64),
            "sf_vals": f["static_gratings_summary/design/spatial_frequency_values_cpd"][:].astype(np.float64),
            "phase_vals": f["static_gratings_summary/design/phase_values"][:].astype(np.float64),
            "cond_idx": f["static_gratings_summary/design/cond_idx"][:].astype(np.int32),
            "cond_flat": f["static_gratings_summary/design/cond_flat"][:].astype(np.int32),

            "cond_mean_baseline": f["static_gratings_summary/summary/cond_mean_baseline"][:].astype(np.float64),
            "cond_mean_response": f["static_gratings_summary/summary/cond_mean_response"][:].astype(np.float64),
            "cond_mean_delta": f["static_gratings_summary/summary/cond_mean_delta"][:].astype(np.float64),
            "cond_count": f["static_gratings_summary/summary/cond_count"][:].astype(np.int32),
            "cov_delta": f["static_gratings_summary/summary/cov_delta"][:].astype(np.float64),
        }
    return out

In [ ]:
# %%
# 07 — SPD and similarity helpers
def symmetrize(M):
    return 0.5 * (M + M.T)

def eigh_clip(M, eps=1e-6):
    w, V = np.linalg.eigh(symmetrize(M))
    w = np.clip(w, eps, None)
    return w, V

def spd_inv(M, eps=1e-6):
    w, V = eigh_clip(M, eps=eps)
    return symmetrize((V * (1.0 / w)) @ V.T)

def spd_invsqrt(M, eps=1e-6):
    w, V = eigh_clip(M, eps=eps)
    return symmetrize((V * (1.0 / np.sqrt(w))) @ V.T)

def airm_distance_np(A, B, eps=1e-6):
    A_inv_sqrt = spd_invsqrt(A, eps=eps)
    C = symmetrize(A_inv_sqrt @ B @ A_inv_sqrt)
    w, _ = eigh_clip(C, eps=eps)
    return float(np.sqrt(np.sum(np.log(w) ** 2)))

def sras_np(A, B, eps=1e-6):
    return float(np.exp(-airm_distance_np(A, B, eps=eps) / np.sqrt(A.shape[0])))

def gamma_and_shape(G, eps=1e-12):
    G = symmetrize(np.asarray(G, dtype=np.float64))
    tr = float(np.trace(G))
    k = G.shape[0]

    gamma = tr / k

    if tr <= eps:
        # Safe fallback; the paper's shape-only object is undefined at zero trace.
        G_shape = np.eye(k, dtype=np.float64) / k
    else:
        G_shape = symmetrize(G / tr)

    return gamma, G_shape
def normalized_coupling(G, i, j, eps=1e-8):
    return float(G[i, j] / np.sqrt(max(G[i, i], eps) * max(G[j, j], eps)))

In [ ]:
# %%
# 08 — Condition means and pooled covariance
def compute_condition_means_and_counts(X, cond_idx, ori_vals, sf_vals, phase_vals):
    O, S, P = len(ori_vals), len(sf_vals), len(phase_vals)
    C = X.shape[1]

    mu = np.full((O, S, P, C), np.nan, dtype=np.float64)
    counts = np.zeros((O, S, P), dtype=np.int32)

    for oi in range(O):
        for si in range(S):
            for pi in range(P):
                mask = (
                    (cond_idx[:, 0] == oi) &
                    (cond_idx[:, 1] == si) &
                    (cond_idx[:, 2] == pi)
                )
                Xi = X[mask]
                counts[oi, si, pi] = Xi.shape[0]
                if Xi.shape[0] > 0:
                    mu[oi, si, pi] = Xi.mean(axis=0)

    return mu, counts

def compute_pooled_cov_delta(delta_trials, cond_idx):
    residuals = []
    for c in np.unique(cond_idx, axis=0):
        mask = np.all(cond_idx == c[None, :], axis=1)
        Xi = delta_trials[mask].astype(np.float64)
        if Xi.shape[0] < 2:
            continue
        residuals.append(Xi - Xi.mean(axis=0, keepdims=True))

    if len(residuals) == 0:
        C = delta_trials.shape[1]
        return np.eye(C, dtype=np.float64), np.nan

    residuals = np.concatenate(residuals, axis=0)
    lw = LedoitWolf(assume_centered=True)
    lw.fit(residuals)
    return lw.covariance_.astype(np.float64), float(lw.shrinkage_)

In [ ]:
# %%
# 09 — Core geometry estimator: use the older finite-difference logic
def compute_local_geometry_from_condmean(mu, cov, ori_vals, sf_vals, phase_vals, eps=1e-6):
    O, S, P, C = mu.shape

    dtheta = np.deg2rad(float(ori_vals[1] - ori_vals[0]))
    rho = np.log2(sf_vals.astype(np.float64))
    dphase = float(phase_vals[1] - phase_vals[0])

    inv_cov = spd_inv(cov + eps * np.eye(cov.shape[0]), eps=eps)

    I_list = []
    M_list = []

    for oi in range(O):
        om = (oi - 1) % O
        op = (oi + 1) % O
        for si in range(1, S - 1):
            for pi in range(P):
                pm = (pi - 1) % P
                pp = (pi + 1) % P

                d_ori = (mu[op, si, pi] - mu[om, si, pi]) / (2.0 * dtheta)
                d_sf = (mu[oi, si + 1, pi] - mu[oi, si - 1, pi]) / (rho[si + 1] - rho[si - 1])
                d_phase = (mu[oi, si, pp] - mu[oi, si, pm]) / (2.0 * dphase)

                A = np.stack([d_ori, d_sf, d_phase], axis=1).astype(np.float64)
                Iu = symmetrize(A.T @ inv_cov @ A)
                Mu = symmetrize(A.T @ A)

                I_list.append(Iu)
                M_list.append(Mu)

    G = symmetrize(np.stack(I_list, axis=0).mean(axis=0))
    G_naive = symmetrize(np.stack(M_list, axis=0).mean(axis=0))

    return {
        "G": G,
        "G_naive": G_naive,
    }

In [ ]:
# %%
# 10 — Family ladder summaries
FAMILY_LADDER = {
    "theta": [0],
    "rho": [1],
    "phi": [2],
    "theta_rho": [0, 1],
    "theta_phi": [0, 2],
    "rho_phi": [1, 2],
    "theta_rho_phi": [0, 1, 2],
}
FAMILY_NAMES = list(FAMILY_LADDER.keys())

def extract_family_matrix(G, family_name):
    idx = FAMILY_LADDER[family_name]
    return symmetrize(G[np.ix_(idx, idx)])

def family_summary_from_G(G, eps=1e-6):
    out = {}

    for fam, idx in FAMILY_LADDER.items():
        Gf = extract_family_matrix(G, fam)
        out[f"{fam}__G"] = Gf
        out[f"{fam}__trace"] = float(np.trace(Gf))
        out[f"{fam}__gamma"] = gamma_and_shape(Gf, eps=eps)[0]

        if len(idx) >= 2:
            out[f"{fam}__G_det_shape"] = gamma_and_shape(Gf, eps=eps)[1]

        if fam == "theta_rho_phi":
            diag = np.diag(Gf)
            tr = np.trace(Gf)
            out["theta_frac"] = float(diag[0] / tr)
            out["rho_frac"] = float(diag[1] / tr)
            out["phi_frac"] = float(diag[2] / tr)
            out["kappa_theta_rho"] = normalized_coupling(Gf, 0, 1)
            out["kappa_theta_phi"] = normalized_coupling(Gf, 0, 2)
            out["kappa_rho_phi"] = normalized_coupling(Gf, 1, 2)

    return out

In [ ]:
# %%
# 11 — Decoders and mapping baselines
def linear_cka(X, Y, eps=1e-12):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y, dtype=np.float64)
    X = X - X.mean(axis=0, keepdims=True)
    Y = Y - Y.mean(axis=0, keepdims=True)
    XtY = X.T @ Y
    XtX = X.T @ X
    YtY = Y.T @ Y
    hsic = np.linalg.norm(XtY, ord="fro") ** 2
    xx = np.linalg.norm(XtX, ord="fro")
    yy = np.linalg.norm(YtY, ord="fro")
    return float(hsic / max(xx * yy, eps))

def rsa_correlation(X, Y):
    DX = squareform(pdist(X, metric="euclidean"))
    DY = squareform(pdist(Y, metric="euclidean"))
    iu = np.triu_indices_from(DX, k=1)
    from scipy.stats import spearmanr
    return float(spearmanr(DX[iu], DY[iu]).correlation)

def cv_accuracy(X, y, seed=0):
    y = np.asarray(y)
    classes, counts = np.unique(y, return_counts=True)
    n_splits = min(5, int(counts.min()))
    if n_splits < 2:
        return np.nan

    clf = make_pipeline(
        StandardScaler(with_mean=True, with_std=True),
        LogisticRegression(max_iter=2000, solver="lbfgs"),
    )
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return float(np.mean(cross_val_score(clf, X, y, cv=cv, scoring="accuracy")))

def build_decoder_profile(cond_mean):
    X = cond_mean.reshape(-1, cond_mean.shape[-1])

    O, S, P = cond_mean.shape[:3]
    y_theta = np.repeat(np.arange(O), S * P)
    y_rho = np.tile(np.repeat(np.arange(S), P), O)
    y_phi = np.tile(np.arange(P), O * S)
    y_full = y_theta * (S * P) + y_rho * P + y_phi

    return np.array([
        cv_accuracy(X, y_theta, seed=CFG["SEED"]),
        cv_accuracy(X, y_rho, seed=CFG["SEED"]),
        cv_accuracy(X, y_phi, seed=CFG["SEED"]),
        cv_accuracy(X, y_full, seed=CFG["SEED"]),
    ], dtype=np.float64)

def mapping_score(Xs, Xt, method="ridge", n_splits=10, seed=0):
    rng = np.random.default_rng(seed)
    n = Xs.shape[0]
    scores = []

    for _ in range(n_splits):
        perm = rng.permutation(n)
        tr = perm[: n // 2]
        te = perm[n // 2 :]

        Xtr, Xte = Xs[tr], Xs[te]
        Ytr, Yte = Xt[tr], Xt[te]

        if method == "ridge":
            model = RidgeCV(alphas=CFG["RIDGE_ALPHAS"])
        else:
            n_comp = min(CFG["PLS_N_COMPONENTS_MAX"], Xtr.shape[1], Ytr.shape[1], len(tr) - 1)
            n_comp = max(1, n_comp)
            model = PLSRegression(n_components=n_comp)

        model.fit(Xtr, Ytr)
        Yhat = model.predict(Xte)

        corrs = []
        for j in range(Yte.shape[1]):
            yt = Yte[:, j]
            yh = Yhat[:, j]
            if np.std(yt) < 1e-8 or np.std(yh) < 1e-8:
                continue
            corrs.append(np.corrcoef(yt, yh)[0, 1])

        scores.append(np.nanmean(corrs) if len(corrs) else np.nan)

    return float(np.nanmean(scores))

In [ ]:
# %%
# 11b — Decoder profile helper (fixed: drop impossible 120-way task)
def build_decoder_profile(cond_mean):
    X = cond_mean.reshape(-1, cond_mean.shape[-1])

    O, S, P = cond_mean.shape[:3]
    y_theta = np.repeat(np.arange(O), S * P)
    y_rho = np.tile(np.repeat(np.arange(S), P), O)
    y_phi = np.tile(np.arange(P), O * S)

    return np.array([
        cv_accuracy(X, y_theta, seed=CFG["SEED"]),
        cv_accuracy(X, y_rho, seed=CFG["SEED"]),
        cv_accuracy(X, y_phi, seed=CFG["SEED"]),
    ], dtype=np.float64)

In [ ]:
# %%
# 12 — Cache builder (cohort-safe)

def cache_path(exp_id, cohort_name, n_match, notebook_version=None):
    if notebook_version is None:
        notebook_version = CFG["NOTEBOOK_VERSION"]

    cohort_dir = CACHE_DIR / f"cohort_{cohort_name}"
    cohort_dir.mkdir(parents=True, exist_ok=True)

    return cohort_dir / (
        f"analysis_exp_{int(exp_id)}"
        f"__cohort_{str(cohort_name)}"
        f"__nmatch_{int(n_match)}"
        f"__ver_{str(notebook_version)}.npz"
    )


def split_half_reliability(delta_trials, cond_idx, ori_vals, sf_vals, phase_vals, repeats=8, seed=0):
    rng = np.random.default_rng(seed)
    scores = []

    uniq = np.unique(cond_idx, axis=0)
    cond_to_trials = []
    for c in uniq:
        idx = np.where(np.all(cond_idx == c[None, :], axis=1))[0]
        cond_to_trials.append(idx)

    for _ in range(repeats):
        idx1, idx2 = [], []
        for idx in cond_to_trials:
            perm = rng.permutation(idx)
            half = len(perm) // 2
            idx1.append(perm[:half])
            idx2.append(perm[half:2 * half])

        idx1 = np.concatenate(idx1)
        idx2 = np.concatenate(idx2)

        mu1, _ = compute_condition_means_and_counts(
            delta_trials[idx1], cond_idx[idx1], ori_vals, sf_vals, phase_vals
        )
        mu2, _ = compute_condition_means_and_counts(
            delta_trials[idx2], cond_idx[idx2], ori_vals, sf_vals, phase_vals
        )
        cov1, _ = compute_pooled_cov_delta(delta_trials[idx1], cond_idx[idx1])
        cov2, _ = compute_pooled_cov_delta(delta_trials[idx2], cond_idx[idx2])

        G1 = compute_local_geometry_from_condmean(
            mu1, cov1, ori_vals, sf_vals, phase_vals, eps=CFG["EPS_SPD"]
        )["G"]
        G2 = compute_local_geometry_from_condmean(
            mu2, cov2, ori_vals, sf_vals, phase_vals, eps=CFG["EPS_SPD"]
        )["G"]

        scores.append(sras_np(G1, G2, eps=CFG["EPS_SPD"]))

    return np.asarray(scores, dtype=np.float64)


def build_analysis_cache(row_dict, cohort_name, n_match):
    exp_id = int(row_dict["id"])
    out_path = cache_path(exp_id, cohort_name=cohort_name, n_match=n_match)

    if out_path.exists() and not CFG["FORCE_REBUILD_CACHE"]:
        return {
            "id": exp_id,
            "cohort_name": cohort_name,
            "status": "exists",
            "path": str(out_path),
        }

    b = load_static_bundle(row_dict["path_h5"])
    delta_trials = b["delta_trials"]
    cond_idx = b["cond_idx"]
    ori_vals = b["ori_vals"]
    sf_vals = b["sf_vals"]
    phase_vals = b["phase_vals"]
    n_cells = delta_trials.shape[1]

    geom_full = compute_local_geometry_from_condmean(
        b["cond_mean_delta"], b["cov_delta"], ori_vals, sf_vals, phase_vals, eps=CFG["EPS_SPD"]
    )

    split_half_full = split_half_reliability(
        delta_trials,
        cond_idx,
        ori_vals,
        sf_vals,
        phase_vals,
        repeats=CFG["SPLIT_HALF_REPEATS_FULL"],
        seed=CFG["SEED"] + exp_id,
    )

    rng = np.random.default_rng(CFG["SEED"] + exp_id)
    G_sub, G_naive_sub, split_half_matched = [], [], []

    for k in range(CFG["N_SUBSAMPLES"]):
        cells = rng.choice(n_cells, size=n_match, replace=False)
        X_sub = delta_trials[:, cells]

        mu_sub, _ = compute_condition_means_and_counts(
            X_sub, cond_idx, ori_vals, sf_vals, phase_vals
        )
        cov_sub, _ = compute_pooled_cov_delta(X_sub, cond_idx)

        geom_sub = compute_local_geometry_from_condmean(
            mu_sub, cov_sub, ori_vals, sf_vals, phase_vals, eps=CFG["EPS_SPD"]
        )

        G_sub.append(geom_sub["G"])
        G_naive_sub.append(geom_sub["G_naive"])

        sh = split_half_reliability(
            X_sub,
            cond_idx,
            ori_vals,
            sf_vals,
            phase_vals,
            repeats=CFG["SPLIT_HALF_REPEATS_MATCHED"],
            seed=CFG["SEED"] + exp_id + k,
        )
        split_half_matched.append(sh.mean())

    G_sub = np.stack(G_sub, axis=0)
    G_naive_sub = np.stack(G_naive_sub, axis=0)
    split_half_matched = np.asarray(split_half_matched, dtype=np.float64)

    decoder_profile = (
        build_decoder_profile(b["cond_mean_delta"])
        if CFG["RUN_DECODER_BASELINES"]
        else np.full((3,), np.nan)
    )
    X120_full = b["cond_mean_delta"].reshape(-1, b["cond_mean_delta"].shape[-1]).astype(np.float64)

    payload = {
        "id": exp_id,
        "targeted_structure": row_dict["targeted_structure"],
        "donor_name": row_dict["donor_name"],
        "imaging_depth": row_dict["imaging_depth"],
        "cre_line": row_dict["cre_line"],
        "visp_layer_label": row_dict.get("visp_layer_label", ""),
        "n_cells": int(n_cells),

        "cache_cohort_name": str(cohort_name),
        "n_match_used": int(n_match),

        "X120_full": X120_full,
        "decoder_profile": decoder_profile,

        "G_full": geom_full["G"],
        "G_naive_full": geom_full["G_naive"],
        "G_matched_mean": G_sub.mean(axis=0),
        "G_naive_matched_mean": G_naive_sub.mean(axis=0),

        "split_half_mean_full": float(split_half_full.mean()),
        "split_half_mean_matched": float(split_half_matched.mean()),
    }

    payload.update({f"full__{k}": v for k, v in family_summary_from_G(payload["G_matched_mean"]).items()})
    payload.update({f"naive__{k}": v for k, v in family_summary_from_G(payload["G_naive_matched_mean"]).items()})

    # always-present state fields
    payload["state_ok"] = False
    payload["n_still"] = 0
    payload["n_run"] = 0

    if CFG["RUN_OPTIONAL_STATE_MODULE"]:
        speeds = b["trial_running_speed"]
        still_mask = speeds <= CFG["RUN_SPEED_STILL_MAX"]
        run_mask = speeds >= CFG["RUN_SPEED_RUN_MIN"]

        payload["n_still"] = int(still_mask.sum())
        payload["n_run"] = int(run_mask.sum())

        uniq = np.unique(cond_idx, axis=0)
        still_min = min(np.sum(np.all(cond_idx == c[None, :], axis=1) & still_mask) for c in uniq)
        run_min = min(np.sum(np.all(cond_idx == c[None, :], axis=1) & run_mask) for c in uniq)

        if (
            payload["n_still"] >= CFG["MIN_STATE_TOTAL_TRIALS"]
            and payload["n_run"] >= CFG["MIN_STATE_TOTAL_TRIALS"]
            and still_min >= CFG["MIN_STATE_TRIALS_PER_CONDITION"]
            and run_min >= CFG["MIN_STATE_TRIALS_PER_CONDITION"]
        ):
            mu_still, _ = compute_condition_means_and_counts(
                delta_trials[still_mask], cond_idx[still_mask], ori_vals, sf_vals, phase_vals
            )
            cov_still, _ = compute_pooled_cov_delta(delta_trials[still_mask], cond_idx[still_mask])

            mu_run, _ = compute_condition_means_and_counts(
                delta_trials[run_mask], cond_idx[run_mask], ori_vals, sf_vals, phase_vals
            )
            cov_run, _ = compute_pooled_cov_delta(delta_trials[run_mask], cond_idx[run_mask])

            G_still = compute_local_geometry_from_condmean(
                mu_still, cov_still, ori_vals, sf_vals, phase_vals, eps=CFG["EPS_SPD"]
            )["G"]
            G_run = compute_local_geometry_from_condmean(
                mu_run, cov_run, ori_vals, sf_vals, phase_vals, eps=CFG["EPS_SPD"]
            )["G"]

            payload["state_ok"] = True
            payload["G_still"] = G_still
            payload["G_run"] = G_run
            payload.update({f"still__{k}": v for k, v in family_summary_from_G(G_still).items()})
            payload.update({f"run__{k}": v for k, v in family_summary_from_G(G_run).items()})

    np.savez_compressed(out_path, payload=np.array([payload], dtype=object))

    return {
        "id": exp_id,
        "cohort_name": cohort_name,
        "status": "ok",
        "path": str(out_path),
    }

In [ ]:
# %%
# 13 — Rebuild caches from scratch with cohort-safe schema
import shutil

shutil.rmtree(CACHE_DIR, ignore_errors=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def build_worker(row_dict, cohort_name, n_match):
    try:
        return build_analysis_cache(row_dict, cohort_name, n_match)
    except Exception as e:
        return {
            "id": int(row_dict["id"]),
            "cohort_name": cohort_name,
            "status": "error",
            "error": repr(e),
        }

area_build = Parallel(
    n_jobs=CFG["N_JOBS_BUILD"],
    backend="loky",
    batch_size=1,
    verbose=10,
)(
    delayed(build_worker)(row, "area", N_MATCH_AREA)
    for row in area_df.to_dict(orient="records")
)

visp_build = Parallel(
    n_jobs=CFG["N_JOBS_BUILD"],
    backend="loky",
    batch_size=1,
    verbose=10,
)(
    delayed(build_worker)(row, "visp", N_MATCH_VISP)
    for row in visp_df.to_dict(orient="records")
)

area_build_df = pd.DataFrame(area_build)
visp_build_df = pd.DataFrame(visp_build)

area_build_df.to_csv(LOG_DIR / "area_cache_build_log.csv", index=False)
visp_build_df.to_csv(LOG_DIR / "visp_cache_build_log.csv", index=False)

print("Area build status:")
display(area_build_df["status"].value_counts(dropna=False))
if "error" in area_build_df["status"].values:
    display(area_build_df[area_build_df["status"] == "error"].head(20))

print("VISp build status:")
display(visp_build_df["status"].value_counts(dropna=False))
if "error" in visp_build_df["status"].values:
    display(visp_build_df[visp_build_df["status"] == "error"].head(20))

In [ ]:
# %%
# 14 — Assert caches exist before loading
missing_area = [
    int(x) for x in area_df["id"]
    if not cache_path(int(x), cohort_name="area", n_match=N_MATCH_AREA).exists()
]
missing_visp = [
    int(x) for x in visp_df["id"]
    if not cache_path(int(x), cohort_name="visp", n_match=N_MATCH_VISP).exists()
]

print("Missing area caches:", len(missing_area))
print("Missing VISp caches:", len(missing_visp))

if len(missing_area):
    print("First missing area ids:", missing_area[:10])
if len(missing_visp):
    print("First missing VISp ids:", missing_visp[:10])

assert len(missing_area) == 0 and len(missing_visp) == 0, (
    "Some analysis caches are missing. Check Cell 13 build logs."
)

In [ ]:
# %%
# 15 — Load caches and attach cohort labels after loading
def load_cache_payload(exp_id, cohort_name, n_match):
    z = np.load(
        cache_path(exp_id, cohort_name=cohort_name, n_match=n_match),
        allow_pickle=True,
    )
    obj = z["payload"][0]
    return obj.item() if hasattr(obj, "item") else obj

def build_cache_df(source_df, cohort_name, n_match):
    rows = []
    for row in source_df.itertuples(index=False):
        rec = load_cache_payload(int(row.id), cohort_name=cohort_name, n_match=n_match)
        rec["cohort_name"] = cohort_name
        rec["cohort_label"] = row.label
        rows.append(rec)
    return pd.DataFrame(rows)

area_cache_df = build_cache_df(area_df, "area", N_MATCH_AREA)
visp_cache_df = build_cache_df(visp_df, "visp", N_MATCH_VISP)

print("Area experiments:", len(area_cache_df))
print("VISp layer experiments:", len(visp_cache_df))
display(area_cache_df.head())
display(visp_cache_df.head())
# %%
# 15b — Hard contamination checks
print("Area labels:")
print(sorted(area_cache_df["cohort_label"].astype(str).unique()))

print("\nVISp labels:")
print(sorted(visp_cache_df["cohort_label"].astype(str).unique()))

print("\nArea n_match_used counts:")
print(area_cache_df["n_match_used"].value_counts(dropna=False))

print("\nVISp n_match_used counts:")
print(visp_cache_df["n_match_used"].value_counts(dropna=False))

print("\nArea VISp rows label counts:")
print(
    area_cache_df[
        area_cache_df["targeted_structure"].astype(str) == "VISp"
    ]["cohort_label"].value_counts(dropna=False)
)

assert set(area_cache_df["cohort_label"].astype(str).unique()) == {
    "VISp", "VISl", "VISal", "VISpm", "VISam", "VISrl"
}
assert set(visp_cache_df["cohort_label"].astype(str).unique()) == {
    "L2/3", "L4", "L5", "L6"
}
assert set(area_cache_df["n_match_used"].astype(int).unique()) == {int(N_MATCH_AREA)}
assert set(visp_cache_df["n_match_used"].astype(int).unique()) == {int(N_MATCH_VISP)}
assert set(
    area_cache_df[
        area_cache_df["targeted_structure"].astype(str) == "VISp"
    ]["cohort_label"].astype(str).unique()
) == {"VISp"}

print("\nContamination checks passed.")

In [ ]:
print("Area labels:", sorted(area_cache_df["cohort_label"].astype(str).unique()))
print("VISp labels:", sorted(visp_cache_df["cohort_label"].astype(str).unique()))

print("\nArea VISp rows label counts:")
print(
    area_cache_df[area_cache_df["targeted_structure"].astype(str) == "VISp"]["cohort_label"]
    .value_counts(dropna=False)
)

assert set(area_cache_df["cohort_label"].astype(str).unique()) == {"VISp", "VISl", "VISal", "VISpm", "VISam", "VISrl"}
assert set(visp_cache_df["cohort_label"].astype(str).unique()) == {"L2/3", "L4", "L5", "L6"}
assert set(area_cache_df["n_match_used"].astype(int).unique()) == {int(N_MATCH_AREA)}
assert set(visp_cache_df["n_match_used"].astype(int).unique()) == {int(N_MATCH_VISP)}
assert set(
    area_cache_df[area_cache_df["targeted_structure"].astype(str) == "VISp"]["cohort_label"].astype(str).unique()
) == {"VISp"}

In [ ]:
# %%
# 16 — Count control and reliability
def safe_corr(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return np.nan
    return float(np.corrcoef(x[m], y[m])[0, 1])


def enrich_for_count_control(df):
    out = df.copy()

    out["gamma_full"] = [gamma_and_shape(G)[0] for G in out["G_full"]]
    out["gamma_matched"] = [gamma_and_shape(G)[0] for G in out["G_matched_mean"]]
    out["gamma_naive_matched"] = [gamma_and_shape(G)[0] for G in out["G_naive_matched_mean"]]

    # use the cohort-safe cache field
    out["overlap_fraction_proxy"] = (
        np.minimum(out["n_match_used"], out["n_cells"]).astype(float)
        / out["n_cells"].astype(float)
    )

    return out


area_cache_df = enrich_for_count_control(area_cache_df)
visp_cache_df = enrich_for_count_control(visp_cache_df)


def count_control_summary(cache_df, cohort_name):
    return {
        "cohort": cohort_name,
        "n_experiments": int(len(cache_df)),
        "n_match": int(cache_df["n_match_used"].iloc[0]),
        "mean_matched_reliability": float(np.nanmean(cache_df["split_half_mean_matched"])),
        "corr_full_gamma_ncells": safe_corr(cache_df["gamma_full"], cache_df["n_cells"]),
        "corr_matched_gamma_ncells": safe_corr(cache_df["gamma_matched"], cache_df["n_cells"]),
        "corr_naive_matched_gamma_ncells": safe_corr(cache_df["gamma_naive_matched"], cache_df["n_cells"]),
        "corr_matched_reliability_ncells": safe_corr(cache_df["split_half_mean_matched"], cache_df["n_cells"]),
        "corr_matched_reliability_overlap_proxy": safe_corr(
            cache_df["split_half_mean_matched"],
            cache_df["overlap_fraction_proxy"],
        ),
    }


count_control_df = pd.DataFrame([
    count_control_summary(area_cache_df, "area"),
    count_control_summary(visp_cache_df, "visp"),
])

count_control_df.to_csv(TABLE_DIR / "count_control_summary.csv", index=False)
display(count_control_df)

In [ ]:
# %%
# 17 — Pairwise similarity table
def finite_profile_corr(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 2:
        return np.nan
    a = a[m] - a[m].mean()
    b = b[m] - b[m].mean()
    den = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / den) if den > 1e-12 else np.nan

def build_pairwise_method_table(cache_df, cohort_name):
    rows = []
    for i in range(len(cache_df)):
        ri = cache_df.iloc[i]
        Xi = ri["X120_full"]
        dpi = ri["decoder_profile"]

        for j in range(i + 1, len(cache_df)):
            rj = cache_df.iloc[j]
            if ri["donor_name"] == rj["donor_name"]:
                continue

            Xj = rj["X120_full"]
            dpj = rj["decoder_profile"]

            rows.append({
                "cohort": cohort_name,
                "id_i": int(ri["id"]),
                "id_j": int(rj["id"]),
                "label_i": ri["cohort_label"],
                "label_j": rj["cohort_label"],
                "donor_i": ri["donor_name"],
                "donor_j": rj["donor_name"],

                "full_sras": sras_np(ri["full__theta_rho_phi__G"], rj["full__theta_rho_phi__G"], eps=CFG["EPS_SPD"]),
                "shape_sras": sras_np(ri["full__theta_rho_phi__G_det_shape"], rj["full__theta_rho_phi__G_det_shape"], eps=CFG["EPS_SPD"]),
                "naive_full_sras": sras_np(ri["naive__theta_rho_phi__G"], rj["naive__theta_rho_phi__G"], eps=CFG["EPS_SPD"]),
                "naive_shape_sras": sras_np(ri["naive__theta_rho_phi__G_det_shape"], rj["naive__theta_rho_phi__G_det_shape"], eps=CFG["EPS_SPD"]),

                "cka": linear_cka(Xi, Xj),
                "rsa": rsa_correlation(Xi, Xj),
                "decoder_profile_corr": finite_profile_corr(dpi, dpj),
            })
    return pd.DataFrame(rows)

area_pair_df = build_pairwise_method_table(area_cache_df, "area")
visp_pair_df = build_pairwise_method_table(visp_cache_df, "visp")

area_pair_df.to_csv(TABLE_DIR / "area_pairwise_similarity.csv", index=False)
visp_pair_df.to_csv(TABLE_DIR / "visp_pairwise_similarity.csv", index=False)
display(area_pair_df.head())

In [ ]:
# %%
# 18 — Mapping baselines (highly parallel JAX ridge + tqdm progress bar)

# %%
# JAX ridge + progress-bar helpers for mapping baselines

import joblib

CFG.setdefault("RUN_PLS_BASELINE", True)
CFG.setdefault("MAP_BATCH_SIZE", 1)

@contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_callback
        tqdm_object.close()


def _standardize_from_train_np(train, test, eps=1e-8):
    mu = train.mean(axis=0, keepdims=True)
    sd = train.std(axis=0, keepdims=True)
    sd = np.where(sd < eps, 1.0, sd)
    return (train - mu) / sd, (test - mu) / sd


def _mean_corr_cols_np(y_true, y_pred, eps=1e-8):
    yt = y_true - y_true.mean(axis=0, keepdims=True)
    yp = y_pred - y_pred.mean(axis=0, keepdims=True)

    num = np.sum(yt * yp, axis=0)
    den = np.sqrt(np.sum(yt ** 2, axis=0) * np.sum(yp ** 2, axis=0))

    valid = den > eps
    if valid.sum() == 0:
        return np.nan
    return float(np.mean(num[valid] / den[valid]))


@jax.jit
def _ridge_gcv_and_test_scores_jax(Ktr, Kte, Ytr_s, Yte_s, alphas):
    """
    Ktr: [n_train, n_train]
    Kte: [n_test,  n_train]
    Ytr_s: [n_train, q]
    Yte_s: [n_test,  q]
    alphas: [A]
    """
    n_train = Ktr.shape[0]
    I = jnp.eye(n_train, dtype=Ktr.dtype)

    def one_alpha(alpha):
        A = Ktr + alpha * I
        C = jnp.linalg.solve(A, Ytr_s)         # [n_train, q]

        Yhat_tr = Ktr @ C
        Yhat_te = Kte @ C

        resid = Ytr_s - Yhat_tr
        rss = jnp.mean(resid ** 2)

        # hat-matrix trace for GCV
        H = jnp.linalg.solve(A, Ktr)
        df = jnp.trace(H)
        gcv = rss / jnp.maximum((1.0 - df / n_train) ** 2, 1e-8)

        # mean correlation across target columns
        yt = Yte_s - jnp.mean(Yte_s, axis=0, keepdims=True)
        yp = Yhat_te - jnp.mean(Yhat_te, axis=0, keepdims=True)

        num = jnp.sum(yt * yp, axis=0)
        den = jnp.sqrt(jnp.sum(yt ** 2, axis=0) * jnp.sum(yp ** 2, axis=0))
        valid = den > 1e-8

        corr = jnp.where(valid, num / den, jnp.nan)
        n_valid = jnp.sum(valid)
        mean_corr = jnp.where(n_valid > 0, jnp.nansum(corr) / n_valid, jnp.nan)

        return gcv, mean_corr

    gcv_vals, test_scores = jax.vmap(one_alpha)(alphas)
    return gcv_vals, test_scores


def mapping_score_ridge_jax(Xs, Xt, n_splits=10, seed=0, alphas=None):
    """
    Fast dual-form ridge with JAX solve and GCV alpha selection inside each split.
    Uses no leakage from the held-out test half.
    """
    Xs = np.asarray(Xs, dtype=np.float64)
    Xt = np.asarray(Xt, dtype=np.float64)

    if alphas is None:
        alphas = np.asarray(CFG["RIDGE_ALPHAS"], dtype=np.float64)

    rng = np.random.default_rng(seed)
    n = Xs.shape[0]
    split_scores = []

    for _ in range(n_splits):
        perm = rng.permutation(n)
        n_train = n // 2
        tr = perm[:n_train]
        te = perm[n_train:]

        Xtr, Xte = Xs[tr], Xs[te]
        Ytr, Yte = Xt[tr], Xt[te]

        # standardize from train
        Xtr_s, Xte_s = _standardize_from_train_np(Xtr, Xte)
        Ytr_s, Yte_s = _standardize_from_train_np(Ytr, Yte)

        # dual ridge works in sample-space, so matrices are only ~60x60
        Ktr = Xtr_s @ Xtr_s.T
        Kte = Xte_s @ Xtr_s.T

        gcv_vals, test_scores = _ridge_gcv_and_test_scores_jax(
            jnp.asarray(Ktr),
            jnp.asarray(Kte),
            jnp.asarray(Ytr_s),
            jnp.asarray(Yte_s),
            jnp.asarray(alphas),
        )

        gcv_vals = np.asarray(gcv_vals, dtype=np.float64)
        test_scores = np.asarray(test_scores, dtype=np.float64)

        finite = np.isfinite(gcv_vals)
        if not finite.any():
            split_scores.append(np.nan)
            continue

        best_idx = np.nanargmin(gcv_vals)
        split_scores.append(float(test_scores[best_idx]))

    return float(np.nanmean(split_scores))


def mapping_score_pls_sklearn(Xs, Xt, n_splits=10, seed=0):
    """
    Keep PLS in sklearn. This is usually fine once pairs are parallelized.
    """
    Xs = np.asarray(Xs, dtype=np.float64)
    Xt = np.asarray(Xt, dtype=np.float64)

    rng = np.random.default_rng(seed)
    n = Xs.shape[0]
    scores = []

    for _ in range(n_splits):
        perm = rng.permutation(n)
        n_train = n // 2
        tr = perm[:n_train]
        te = perm[n_train:]

        Xtr, Xte = Xs[tr], Xs[te]
        Ytr, Yte = Xt[tr], Xt[te]

        n_comp = min(CFG["PLS_N_COMPONENTS_MAX"], Xtr.shape[1], Ytr.shape[1], len(tr) - 1)
        n_comp = max(1, n_comp)

        model = PLSRegression(n_components=n_comp)
        model.fit(Xtr, Ytr)
        Yhat = model.predict(Xte)

        scores.append(_mean_corr_cols_np(Yte, Yhat))

    return float(np.nanmean(scores))


def _prepare_mapping_records(cache_df):
    records = []
    for r in cache_df.itertuples(index=False):
        records.append({
            "id": int(r.id),
            "label": r.label if "label" in cache_df.columns else None,
            "donor_name": r.donor_name,
            "X120_full": np.asarray(r.X120_full, dtype=np.float64),
        })
    return records


def _pair_tasks(records, cohort_name):
    tasks = []
    for i in range(len(records)):
        for j in range(i + 1, len(records)):
            if records[i]["donor_name"] == records[j]["donor_name"]:
                continue
            tasks.append((cohort_name, i, j))
    return tasks


def _mapping_pair_worker(records, cohort_name, i, j):
    ri = records[i]
    rj = records[j]

    Xi = ri["X120_full"]
    Xj = rj["X120_full"]

    ridge_ij = mapping_score_ridge_jax(
        Xi, Xj,
        n_splits=CFG["MAP_N_SPLITS"],
        seed=CFG["SEED"],
        alphas=np.asarray(CFG["RIDGE_ALPHAS"], dtype=np.float64),
    )
    ridge_ji = mapping_score_ridge_jax(
        Xj, Xi,
        n_splits=CFG["MAP_N_SPLITS"],
        seed=CFG["SEED"],
        alphas=np.asarray(CFG["RIDGE_ALPHAS"], dtype=np.float64),
    )

    out = {
        "cohort": cohort_name,
        "id_i": int(ri["id"]),
        "id_j": int(rj["id"]),
        "ridge_score": float(np.nanmean([ridge_ij, ridge_ji])),
    }

    if CFG["RUN_PLS_BASELINE"]:
        pls_ij = mapping_score_pls_sklearn(
            Xi, Xj,
            n_splits=CFG["MAP_N_SPLITS"],
            seed=CFG["SEED"],
        )
        pls_ji = mapping_score_pls_sklearn(
            Xj, Xi,
            n_splits=CFG["MAP_N_SPLITS"],
            seed=CFG["SEED"],
        )
        out["pls_score"] = float(np.nanmean([pls_ij, pls_ji]))
    else:
        out["pls_score"] = np.nan

    return out


def build_mapping_table_parallel_jax(cache_df, cohort_name, n_jobs=None):
    if n_jobs is None:
        n_jobs = CFG["N_JOBS_MAP"]

    records = _prepare_mapping_records(cache_df)
    tasks = _pair_tasks(records, cohort_name)

    print(f"{cohort_name}: {len(tasks)} eligible pairs")

    if len(tasks) == 0:
        return pd.DataFrame(columns=["cohort", "id_i", "id_j", "ridge_score", "pls_score"])

    with tqdm_joblib(tqdm(total=len(tasks), desc=f"{cohort_name} mapping", unit="pair")):
        rows = Parallel(
            n_jobs=n_jobs,
            backend="loky",
            batch_size=CFG["MAP_BATCH_SIZE"],
            verbose=0,
        )(
            delayed(_mapping_pair_worker)(records, cohort_name, i, j)
            for cohort_name, i, j in tasks
        )

    return pd.DataFrame(rows)

if CFG["RUN_MAPPING_BASELINES"]:
    area_map_df = build_mapping_table_parallel_jax(
        area_cache_df,
        "area",
        n_jobs=CFG["N_JOBS_MAP"],
    )
    visp_map_df = build_mapping_table_parallel_jax(
        visp_cache_df,
        "visp",
        n_jobs=CFG["N_JOBS_MAP"],
    )
else:
    area_map_df = pd.DataFrame(columns=["cohort", "id_i", "id_j", "ridge_score", "pls_score"])
    visp_map_df = pd.DataFrame(columns=["cohort", "id_i", "id_j", "ridge_score", "pls_score"])

area_map_df.to_csv(TABLE_DIR / "area_mapping_similarity.csv", index=False)
visp_map_df.to_csv(TABLE_DIR / "visp_mapping_similarity.csv", index=False)

display(area_map_df.head())
display(visp_map_df.head())

In [ ]:
# %%
# 19 — Unified query records and headline benchmark
METHOD_COLS = [
    "full_sras",
    "shape_sras",
    "naive_full_sras",
    "naive_shape_sras",
    "cka",
    "rsa",
    "decoder_profile_corr",
]

def merge_pair_tables(pair_df, map_df):
    if len(map_df) == 0:
        return pair_df.copy()
    return pair_df.merge(map_df[["id_i", "id_j", "ridge_score", "pls_score"]], on=["id_i", "id_j"], how="left")

# %%
def melt_pairwise_to_query_records(pair_df):
    rows = []
    cols = [c for c in METHOD_COLS + ["ridge_score", "pls_score"] if c in pair_df.columns]

    has_donor_cols = ("donor_i" in pair_df.columns) and ("donor_j" in pair_df.columns)

    for _, r in pair_df.iterrows():
        for method in cols:
            if pd.isna(r[method]):
                continue

            row_ij = {
                "method": method,
                "query_id": int(r["id_i"]),
                "candidate_id": int(r["id_j"]),
                "query_label": r["label_i"],
                "candidate_label": r["label_j"],
                "same_label": bool(r["label_i"] == r["label_j"]),
                "similarity": float(r[method]),
            }
            row_ji = {
                "method": method,
                "query_id": int(r["id_j"]),
                "candidate_id": int(r["id_i"]),
                "query_label": r["label_j"],
                "candidate_label": r["label_i"],
                "same_label": bool(r["label_i"] == r["label_j"]),
                "similarity": float(r[method]),
            }

            if has_donor_cols:
                row_ij["query_donor"] = r["donor_i"]
                row_ij["candidate_donor"] = r["donor_j"]
                row_ji["query_donor"] = r["donor_j"]
                row_ji["candidate_donor"] = r["donor_i"]

            rows.append(row_ij)
            rows.append(row_ji)

    return pd.DataFrame(rows)

def summarize_method(qr_df, method, cohort_name):
    q = qr_df[(qr_df["method"] == method)].copy()
    if len(q) == 0:
        return None, None

    top_rows = []
    for query_id, g in q.groupby("query_id"):
        by_label = g.groupby("candidate_label")["similarity"].max().reset_index()
        by_label = by_label.sort_values("similarity", ascending=False).reset_index(drop=True)

        pred_label = by_label.iloc[0]["candidate_label"]
        true_label = g["query_label"].iloc[0]

        same_scores = by_label.loc[by_label["candidate_label"] == true_label, "similarity"].to_numpy()
        other_scores = by_label.loc[by_label["candidate_label"] != true_label, "similarity"].to_numpy()

        best_same = float(same_scores.max()) if len(same_scores) else np.nan
        best_other = float(other_scores.max()) if len(other_scores) else np.nan
        margin = float(best_same - best_other) if np.isfinite(best_same) and np.isfinite(best_other) else np.nan

        top_rows.append({
            "cohort": cohort_name,
            "method": method,
            "query_id": int(query_id),
            "query_label": true_label,
            "pred_label": pred_label,
            "correct": int(pred_label == true_label),
            "top_margin": margin,
        })

    top_df = pd.DataFrame(top_rows)
    auc = roc_auc_score(q["same_label"].astype(int), q["similarity"].astype(float))

    return {
        "cohort": cohort_name,
        "method": method,
        "accuracy": float(top_df["correct"].mean()),
        "top_margin_mean": float(top_df["top_margin"].mean()),
        "diagonal_auc": float(auc),
        "n_queries": int(top_df["query_id"].nunique()),
    }, top_df

def summarize_all_methods(qr_df, cohort_name):
    summaries, tops = [], []
    for method in sorted(qr_df["method"].unique()):
        s, t = summarize_method(qr_df, method, cohort_name)
        if s is not None:
            summaries.append(s)
            tops.append(t)
    return pd.DataFrame(summaries).sort_values("accuracy", ascending=False), pd.concat(tops, ignore_index=True)

area_pair_merged = merge_pair_tables(area_pair_df, area_map_df)
visp_pair_merged = merge_pair_tables(visp_pair_df, visp_map_df)

area_qr = melt_pairwise_to_query_records(area_pair_merged)
visp_qr = melt_pairwise_to_query_records(visp_pair_merged)

area_match_df, area_top_df = summarize_all_methods(area_qr, "area")
visp_match_df, visp_top_df = summarize_all_methods(visp_qr, "visp")

area_match_df.to_csv(TABLE_DIR / "area_matching_benchmark.csv", index=False)
visp_match_df.to_csv(TABLE_DIR / "visp_matching_benchmark.csv", index=False)
display(area_match_df)
display(visp_match_df)

In [ ]:
# %%
# 20 — Fisher vs naive headline tests
def paired_permutation_pvalue(a, b, n_perm=5000, seed=0):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    obs = np.mean(a - b)
    rng = np.random.default_rng(seed)
    count = 0
    for _ in range(n_perm):
        signs = rng.integers(0, 2, size=len(a)) * 2 - 1
        stat = np.mean((a - b) * signs)
        if abs(stat) >= abs(obs):
            count += 1
    return float((count + 1) / (n_perm + 1))

def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    order = np.argsort(pvals)
    ranked = pvals[order]
    q = np.empty_like(ranked)
    prev = 1.0
    m = len(ranked)
    for i in range(m - 1, -1, -1):
        prev = min(prev, ranked[i] * m / (i + 1))
        q[i] = prev
    out = np.empty_like(q)
    out[order] = q
    return out

def fisher_vs_naive_tests(top_df, cohort_name):
    comparisons = [
        ("full_sras", "naive_full_sras", "full_vs_naive"),
        ("shape_sras", "naive_shape_sras", "shape_vs_naive"),
        ("full_sras", "cka", "full_vs_cka"),
        ("shape_sras", "cka", "shape_vs_cka"),
    ]

    rows = []
    for m1, m2, label in comparisons:
        a = top_df[top_df["method"] == m1].sort_values("query_id")
        b = top_df[top_df["method"] == m2].sort_values("query_id")
        if len(a) == 0 or len(b) == 0:
            continue

        rows.append({
            "cohort": cohort_name,
            "comparison": label,
            "mean_acc_m1": float(a["correct"].mean()),
            "mean_acc_m2": float(b["correct"].mean()),
            "mean_margin_m1": float(a["top_margin"].mean()),
            "mean_margin_m2": float(b["top_margin"].mean()),
            "perm_p_acc": paired_permutation_pvalue(a["correct"], b["correct"], n_perm=CFG["N_PERM"], seed=CFG["SEED"]),
            "perm_p_margin": paired_permutation_pvalue(a["top_margin"], b["top_margin"], n_perm=CFG["N_PERM"], seed=CFG["SEED"]),
        })

    out = pd.DataFrame(rows)
    if len(out):
        out["q_acc"] = bh_fdr(out["perm_p_acc"].to_numpy())
        out["q_margin"] = bh_fdr(out["perm_p_margin"].to_numpy())
    return out

area_fvn = fisher_vs_naive_tests(area_top_df, "area")
visp_fvn = fisher_vs_naive_tests(visp_top_df, "visp")

display(area_fvn)
display(visp_fvn)

In [ ]:
# %%
# 21 — Family ladder benchmark
def build_family_query_records(cache_df, cohort_name):
    rows = []
    for i in range(len(cache_df)):
        ri = cache_df.iloc[i]
        for j in range(i + 1, len(cache_df)):
            rj = cache_df.iloc[j]
            if ri["donor_name"] == rj["donor_name"]:
                continue

            for fam in FAMILY_NAMES:
                rows.append({
                    "cohort": cohort_name,
                    "family": fam,
                    "variant": "full",
                    "id_i": int(ri["id"]),
                    "id_j": int(rj["id"]),
                    "label_i": ri["cohort_label"],
                    "label_j": rj["cohort_label"],
                    "similarity": sras_np(ri[f"full__{fam}__G"], rj[f"full__{fam}__G"], eps=CFG["EPS_SPD"]),
                })
                if len(FAMILY_LADDER[fam]) >= 2:
                    rows.append({
                        "cohort": cohort_name,
                        "family": fam,
                        "variant": "shape",
                        "id_i": int(ri["id"]),
                        "id_j": int(rj["id"]),
                        "label_i": ri["cohort_label"],
                        "label_j": rj["cohort_label"],
                        "similarity": sras_np(ri[f"full__{fam}__G_det_shape"], rj[f"full__{fam}__G_det_shape"], eps=CFG["EPS_SPD"]),
                    })
    return pd.DataFrame(rows)

area_family_pair = build_family_query_records(area_cache_df, "area")
visp_family_pair = build_family_query_records(visp_cache_df, "visp")

In [ ]:
# %%
# 21b — Grouped-family matching summaries and full-vs-shape tests

def family_pair_to_query_records(family_pair_df):
    rows = []
    for _, r in family_pair_df.iterrows():
        rows.append({
            "query_id": int(r["id_i"]),
            "candidate_id": int(r["id_j"]),
            "query_label": r["label_i"],
            "candidate_label": r["label_j"],
            "same_label": bool(r["label_i"] == r["label_j"]),
            "similarity": float(r["similarity"]),
        })
        rows.append({
            "query_id": int(r["id_j"]),
            "candidate_id": int(r["id_i"]),
            "query_label": r["label_j"],
            "candidate_label": r["label_i"],
            "same_label": bool(r["label_i"] == r["label_j"]),
            "similarity": float(r["similarity"]),
        })
    return pd.DataFrame(rows)

def summarize_family_grouped(family_pair_df, cohort_name):
    summaries = []
    tops = []

    for (family, variant), g in family_pair_df.groupby(["family", "variant"]):
        qr = family_pair_to_query_records(g)

        top_rows = []
        for query_id, qg in qr.groupby("query_id"):
            by_label = qg.groupby("candidate_label")["similarity"].max().reset_index()
            by_label = by_label.sort_values("similarity", ascending=False).reset_index(drop=True)

            pred_label = by_label.iloc[0]["candidate_label"]
            true_label = qg["query_label"].iloc[0]

            same_scores = by_label.loc[by_label["candidate_label"] == true_label, "similarity"].to_numpy()
            other_scores = by_label.loc[by_label["candidate_label"] != true_label, "similarity"].to_numpy()

            best_same = float(same_scores.max()) if len(same_scores) else np.nan
            best_other = float(other_scores.max()) if len(other_scores) else np.nan
            margin = float(best_same - best_other) if np.isfinite(best_same) and np.isfinite(best_other) else np.nan

            top_rows.append({
                "cohort": cohort_name,
                "family": family,
                "variant": variant,
                "query_id": int(query_id),
                "query_label": true_label,
                "pred_label": pred_label,
                "correct": int(pred_label == true_label),
                "top_margin": margin,
            })

        top_df = pd.DataFrame(top_rows)
        auc = roc_auc_score(qr["same_label"].astype(int), qr["similarity"].astype(float))

        summaries.append({
            "cohort": cohort_name,
            "family": family,
            "variant": variant,
            "accuracy": float(top_df["correct"].mean()),
            "top_margin_mean": float(top_df["top_margin"].mean()),
            "diagonal_auc": float(auc),
            "n_queries": int(top_df["query_id"].nunique()),
        })
        tops.append(top_df)

    return pd.DataFrame(summaries), pd.concat(tops, ignore_index=True)

def family_full_vs_shape_table(family_top_df, cohort_name):
    rows = []

    for family in sorted(family_top_df["family"].unique()):
        a = family_top_df[
            (family_top_df["family"] == family) &
            (family_top_df["variant"] == "full")
        ].sort_values("query_id")

        b = family_top_df[
            (family_top_df["family"] == family) &
            (family_top_df["variant"] == "shape")
        ].sort_values("query_id")

        if len(a) == 0 or len(b) == 0:
            continue

        rows.append({
            "cohort": cohort_name,
            "family": family,
            "mean_acc_full": float(a["correct"].mean()),
            "mean_acc_shape": float(b["correct"].mean()),
            "mean_margin_full": float(a["top_margin"].mean()),
            "mean_margin_shape": float(b["top_margin"].mean()),
            "perm_p_acc": paired_permutation_pvalue(
                a["correct"], b["correct"], n_perm=CFG["N_PERM"], seed=CFG["SEED"]
            ),
            "perm_p_margin": paired_permutation_pvalue(
                a["top_margin"], b["top_margin"], n_perm=CFG["N_PERM"], seed=CFG["SEED"]
            ),
        })

    out = pd.DataFrame(rows)
    if len(out):
        out["q_acc"] = bh_fdr(out["perm_p_acc"].to_numpy())
        out["q_margin"] = bh_fdr(out["perm_p_margin"].to_numpy())
    return out

area_family_match_df, area_family_top_df = summarize_family_grouped(area_family_pair, "area")
visp_family_match_df, visp_family_top_df = summarize_family_grouped(visp_family_pair, "visp")

area_full_vs_shape_df = family_full_vs_shape_table(area_family_top_df, "area")
visp_full_vs_shape_df = family_full_vs_shape_table(visp_family_top_df, "visp")

area_family_match_df.to_csv(TABLE_DIR / "area_family_matching.csv", index=False)
visp_family_match_df.to_csv(TABLE_DIR / "visp_family_matching.csv", index=False)
area_full_vs_shape_df.to_csv(TABLE_DIR / "area_family_full_vs_shape.csv", index=False)
visp_full_vs_shape_df.to_csv(TABLE_DIR / "visp_family_full_vs_shape.csv", index=False)

display(area_family_match_df)
display(visp_family_match_df)
display(area_full_vs_shape_df)
display(visp_full_vs_shape_df)

In [ ]:
# %%
# 22 — Incremental value
def build_pair_feature_table(pair_df):
    df = pair_df.copy()
    df["same_label"] = (df["label_i"] == df["label_j"]).astype(int)
    df["pair_group"] = df.apply(lambda r: f"{min(int(r['id_i']), int(r['id_j']))}__{max(int(r['id_i']), int(r['id_j']))}", axis=1)
    return df

def grouped_auc_cv(df, feature_cols):
    use = df.dropna(subset=feature_cols + ["same_label"]).copy()
    if len(use) == 0:
        return np.nan

    X = use[feature_cols].to_numpy(dtype=np.float64)
    y = use["same_label"].to_numpy(dtype=int)
    groups = use["pair_group"].to_numpy()

    n_splits = min(5, len(np.unique(groups)))
    if n_splits < 2:
        return np.nan

    cv = GroupKFold(n_splits=n_splits)
    aucs = []

    for tr, te in cv.split(X, y, groups):
        clf = make_pipeline(
            StandardScaler(with_mean=True, with_std=True),
            LogisticRegression(max_iter=2000, solver="lbfgs"),
        )
        clf.fit(X[tr], y[tr])
        p = clf.predict_proba(X[te])[:, 1]
        aucs.append(roc_auc_score(y[te], p))

    return float(np.mean(aucs))

area_feat = build_pair_feature_table(area_pair_merged)
visp_feat = build_pair_feature_table(visp_pair_merged)

model_specs = {
    "baseline": ["cka", "rsa", "decoder_profile_corr", "ridge_score", "pls_score"],
    "baseline_plus_full": ["cka", "rsa", "decoder_profile_corr", "ridge_score", "pls_score", "full_sras"],
    "baseline_plus_shape": ["cka", "rsa", "decoder_profile_corr", "ridge_score", "pls_score", "shape_sras"],
    "baseline_plus_naive_full": ["cka", "rsa", "decoder_profile_corr", "ridge_score", "pls_score", "naive_full_sras"],
    "baseline_plus_naive_shape": ["cka", "rsa", "decoder_profile_corr", "ridge_score", "pls_score", "naive_shape_sras"],
}

rows = []
for cohort_name, feat_df in [("area", area_feat), ("visp", visp_feat)]:
    for model_name, cols in model_specs.items():
        cols = [c for c in cols if c in feat_df.columns]
        rows.append({
            "cohort": cohort_name,
            "model": model_name,
            "cv_auc": grouped_auc_cv(feat_df, cols),
        })

incremental_df = pd.DataFrame(rows)
incremental_df.to_csv(TABLE_DIR / "incremental_value_models.csv", index=False)
display(incremental_df)

In [ ]:
# %%
# 22b — Incremental value with nuisance controls

def add_pair_nuisance_features(pair_df, cache_df):
    out = pair_df.copy()

    ncells_map = cache_df.set_index("id")["n_cells"].to_dict()
    rel_map = cache_df.set_index("id")["split_half_mean_matched"].to_dict()

    out["n_cells_i"] = out["id_i"].map(ncells_map).astype(float)
    out["n_cells_j"] = out["id_j"].map(ncells_map).astype(float)

    out["log_n_cells_i"] = np.log(out["n_cells_i"])
    out["log_n_cells_j"] = np.log(out["n_cells_j"])
    out["abs_log_n_cells_diff"] = np.abs(out["log_n_cells_i"] - out["log_n_cells_j"])
    out["min_log_n_cells"] = np.minimum(out["log_n_cells_i"], out["log_n_cells_j"])
    out["max_log_n_cells"] = np.maximum(out["log_n_cells_i"], out["log_n_cells_j"])

    out["reliability_i"] = out["id_i"].map(rel_map).astype(float)
    out["reliability_j"] = out["id_j"].map(rel_map).astype(float)
    out["min_reliability"] = np.minimum(out["reliability_i"], out["reliability_j"])
    out["abs_reliability_diff"] = np.abs(out["reliability_i"] - out["reliability_j"])

    return out

area_feat_nuis = add_pair_nuisance_features(area_feat, area_cache_df)
visp_feat_nuis = add_pair_nuisance_features(visp_feat, visp_cache_df)

nuisance_cols = [
    "log_n_cells_i", "log_n_cells_j",
    "abs_log_n_cells_diff", "min_log_n_cells", "max_log_n_cells",
    "reliability_i", "reliability_j",
    "min_reliability", "abs_reliability_diff",
]

nuisance_model_specs = {
    "baseline_plus_nuisance": ["cka", "rsa", "decoder_profile_corr", "ridge_score", "pls_score"] + nuisance_cols,
    "baseline_plus_nuisance_plus_full": ["cka", "rsa", "decoder_profile_corr", "ridge_score", "pls_score", "full_sras"] + nuisance_cols,
    "baseline_plus_nuisance_plus_shape": ["cka", "rsa", "decoder_profile_corr", "ridge_score", "pls_score", "shape_sras"] + nuisance_cols,
    "baseline_plus_nuisance_plus_naive_full": ["cka", "rsa", "decoder_profile_corr", "ridge_score", "pls_score", "naive_full_sras"] + nuisance_cols,
    "baseline_plus_nuisance_plus_naive_shape": ["cka", "rsa", "decoder_profile_corr", "ridge_score", "pls_score", "naive_shape_sras"] + nuisance_cols,
}

rows = []
for cohort_name, feat_df in [("area", area_feat_nuis), ("visp", visp_feat_nuis)]:
    for model_name, cols in nuisance_model_specs.items():
        cols = [c for c in cols if c in feat_df.columns]
        rows.append({
            "cohort": cohort_name,
            "model": model_name,
            "cv_auc": grouped_auc_cv(feat_df, cols),
        })

incremental_nuisance_df = pd.DataFrame(rows)
incremental_nuisance_df.to_csv(TABLE_DIR / "incremental_nuisance_models.csv", index=False)
display(incremental_nuisance_df)

In [ ]:
# %%
# 23 — Area allocation summary
def area_allocation_summary(area_cache_df):
    rows = []
    for _, r in area_cache_df.iterrows():
        rows.append({
            "id": int(r["id"]),
            "label": r["cohort_label"],
            "theta_frac": r["full__theta_frac"],
            "rho_frac": r["full__rho_frac"],
            "phi_frac": r["full__phi_frac"],
            "kappa_theta_rho": r["full__kappa_theta_rho"],
            "kappa_theta_phi": r["full__kappa_theta_phi"],
            "kappa_rho_phi": r["full__kappa_rho_phi"],
        })
    return pd.DataFrame(rows)

area_alloc = area_allocation_summary(area_cache_df)
area_alloc.to_csv(TABLE_DIR / "area_allocation_summary.csv", index=False)
display(area_alloc.head())

In [ ]:
# %%
# 23b — Coupling summaries (experiment-level + by-label)

def coupling_summary_df(cache_df, cohort_name):
    rows = []
    for _, r in cache_df.iterrows():
        rows.append({
            "cohort": cohort_name,
            "id": int(r["id"]),
            "label": r["cohort_label"],
            "theta_frac": float(r["full__theta_frac"]),
            "rho_frac": float(r["full__rho_frac"]),
            "phi_frac": float(r["full__phi_frac"]),
            "kappa_theta_rho": float(r["full__kappa_theta_rho"]),
            "kappa_theta_phi": float(r["full__kappa_theta_phi"]),
            "kappa_rho_phi": float(r["full__kappa_rho_phi"]),
        })
    return pd.DataFrame(rows)

def coupling_by_label_df(exp_df):
    grp = (
        exp_df.groupby("label")[[
            "theta_frac", "rho_frac", "phi_frac",
            "kappa_theta_rho", "kappa_theta_phi", "kappa_rho_phi"
        ]]
        .agg(["mean", "median", "std", "count"])
        .reset_index()
    )
    grp.columns = [
        "__".join(c).strip("_") if isinstance(c, tuple) else c
        for c in grp.columns
    ]
    return grp

area_coupling_df = coupling_summary_df(area_cache_df, "area")
visp_coupling_df = coupling_summary_df(visp_cache_df, "visp")

area_coupling_by_label_df = coupling_by_label_df(area_coupling_df)
visp_coupling_by_label_df = coupling_by_label_df(visp_coupling_df)

area_coupling_df.to_csv(TABLE_DIR / "area_coupling_summary.csv", index=False)
visp_coupling_df.to_csv(TABLE_DIR / "visp_coupling_summary.csv", index=False)
area_coupling_by_label_df.to_csv(TABLE_DIR / "area_coupling_by_label.csv", index=False)
visp_coupling_by_label_df.to_csv(TABLE_DIR / "visp_coupling_by_label.csv", index=False)

display(area_coupling_by_label_df)
display(visp_coupling_by_label_df)

In [ ]:
# %%
# 24 — Optional state diagnostics apparatus (experiment-level, no cross-cohort concat)
if CFG["RUN_OPTIONAL_STATE_MODULE"]:
    state_df = area_cache_df.copy()
    state_df = state_df[state_df["state_ok"].fillna(False)].copy().reset_index(drop=True)

    print("State-valid experiments:", len(state_df))
    display(state_df[["id", "cohort_label", "n_still", "n_run"]].head())
else:
    print("State module disabled.")

In [ ]:
# %%
# 25 — Optional state significance apparatus
if CFG["RUN_OPTIONAL_STATE_MODULE"]:
    def bootstrap_stat(x, n_boot=5000, seed=0):
        x = np.asarray(x, dtype=float)
        rng = np.random.default_rng(seed)
        boots = []
        for _ in range(n_boot):
            sample = rng.choice(x, size=len(x), replace=True)
            boots.append(np.mean(sample))
        boots = np.asarray(boots, dtype=float)
        return float(np.mean(x)), float(np.quantile(boots, 0.025)), float(np.quantile(boots, 0.975))

    def sign_flip_pvalue(x, n_perm=5000, seed=0):
        x = np.asarray(x, dtype=float)
        obs = np.mean(x)
        rng = np.random.default_rng(seed)
        count = 0
        for _ in range(n_perm):
            signs = rng.integers(0, 2, size=len(x)) * 2 - 1
            stat = np.mean(x * signs)
            if abs(stat) >= abs(obs):
                count += 1
        return float((count + 1) / (n_perm + 1))

    state_rows = []
    for _, r in state_df.iterrows():
        for fam in FAMILY_NAMES:
            if f"still__{fam}__G" not in r:
                continue
            Gs = r[f"still__{fam}__G"]
            Gr = r[f"run__{fam}__G"]
            state_rows.append({
                "id": int(r["id"]),
                "targeted_structure": r["targeted_structure"],
                "family": fam,
                "full_run_still_airm": airm_distance_np(Gr, Gs, eps=CFG["EPS_SPD"]),
                "scale_logratio": np.log(gamma_and_shape(Gr, eps=CFG["EPS_SPD"])[0]) - np.log(gamma_and_shape(Gs, eps=CFG["EPS_SPD"])[0]),
            })

    state_summary = pd.DataFrame(state_rows)
    state_summary.to_csv(TABLE_DIR / "state_summary_table.csv", index=False)
    display(state_summary.head())
else:
    print("State module disabled.")

In [ ]:
# %%
# 26 — Optional n_match sweep apparatus (real implementation)

if CFG["RUN_OPTIONAL_NMATCH_SWEEP"]:
    SWEEP_DIR = CACHE_DIR / "nmatch_sweep"
    SWEEP_DIR.mkdir(parents=True, exist_ok=True)

    AREA_SWEEP_VALUES = [30, 45, 60, 90]
    VISP_SWEEP_VALUES = [20, 30, 45, 60]

    def recompute_G_from_subsample(delta_trials, cond_idx, ori_vals, sf_vals, phase_vals, cell_idx):
        X = delta_trials[:, cell_idx]
        mu, _ = compute_condition_means_and_counts(X, cond_idx, ori_vals, sf_vals, phase_vals)
        cov, _ = compute_pooled_cov_delta(X, cond_idx)
        geom = compute_local_geometry_from_condmean(mu, cov, ori_vals, sf_vals, phase_vals, eps=CFG["EPS_SPD"])
        return geom["G"], geom["G_naive"]

    def det_shape(G, eps=1e-6):
        return gamma_and_shape(G, eps=eps)[1]

    def matched_reliability_from_subsamples(G_list):
        if len(G_list) < 2:
            return np.nan
        sims = []
        for i in range(len(G_list)):
            for j in range(i + 1, len(G_list)):
                sims.append(sras_np(G_list[i], G_list[j], eps=CFG["EPS_SPD"]))
        return float(np.nanmean(sims)) if len(sims) else np.nan

    def sweep_cache_path(exp_id, cohort_name, n_match):
        cdir = SWEEP_DIR / f"cohort_{cohort_name}"
        cdir.mkdir(parents=True, exist_ok=True)
        return cdir / f"sweep_exp_{int(exp_id)}__nmatch_{int(n_match)}.npz"

    def build_nmatch_sweep_cache(row_dict, cohort_name, n_match):
        exp_id = int(row_dict["id"])
        out_path = sweep_cache_path(exp_id, cohort_name, n_match)

        if out_path.exists():
            return {"id": exp_id, "n_match": int(n_match), "status": "exists", "cache_path": str(out_path)}

        b = load_static_bundle(row_dict["path_h5"])
        delta_trials = b["delta_trials"]
        cond_idx = b["cond_idx"]
        ori_vals = b["ori_vals"]
        sf_vals = b["sf_vals"]
        phase_vals = b["phase_vals"]

        n_cells = delta_trials.shape[1]
        if n_cells < int(n_match):
            return {"id": exp_id, "n_match": int(n_match), "status": "ineligible", "cache_path": None}

        rng = np.random.default_rng(CFG["SEED"] + exp_id + 1000 * int(n_match))

        Gf_list, Gn_list, Sf_list, Sn_list = [], [], [], []

        for _ in range(int(CFG["N_SUBSAMPLES"])):
            cell_idx = rng.choice(n_cells, size=int(n_match), replace=False)
            Gf, Gn = recompute_G_from_subsample(delta_trials, cond_idx, ori_vals, sf_vals, phase_vals, cell_idx)

            Gf_list.append(Gf)
            Gn_list.append(Gn)
            full_G = np.mean(np.stack(Gf_list, axis=0), axis=0)
            naive_full_G = np.mean(np.stack(Gn_list, axis=0), axis=0)

            payload = {
                "id": exp_id,
                "n_match": int(n_match),
                "full_G": full_G,
                "naive_full_G": naive_full_G,
                "shape_G": gamma_and_shape(full_G, eps=CFG["EPS_SPD"])[1],
                "naive_shape_G": gamma_and_shape(naive_full_G, eps=CFG["EPS_SPD"])[1],
                "matched_reliability_full": matched_reliability_from_subsamples(Gf_list),
            }

        np.savez_compressed(out_path, payload=np.array([payload], dtype=object))
        return {"id": exp_id, "n_match": int(n_match), "status": "ok", "cache_path": str(out_path)}

    def load_sweep_payload(exp_id, cohort_name, n_match):
        z = np.load(sweep_cache_path(exp_id, cohort_name, n_match), allow_pickle=True)
        obj = z["payload"][0]
        return obj.item() if hasattr(obj, "item") else obj

    def build_sweep_df(source_df, cohort_name, n_match):
        eligible = source_df[source_df["n_cells"] >= int(n_match)].copy()

        rows = []
        for row in eligible.itertuples(index=False):
            p = load_sweep_payload(int(row.id), cohort_name, n_match)
            rows.append({
                "id": int(row.id),
                "label": row.label,
                "donor_name": row.donor_name,
                "targeted_structure": row.targeted_structure,
                "visp_layer_label": row.visp_layer_label,
                "n_cells": int(row.n_cells),
                "n_match": int(p["n_match"]),
                "full_G": p["full_G"],
                "shape_G": p["shape_G"],
                "naive_full_G": p["naive_full_G"],
                "naive_shape_G": p["naive_shape_G"],
                "matched_reliability_full": float(p["matched_reliability_full"]),
            })
        return pd.DataFrame(rows)

# %%
# Fix for n-match sweep pair table: include donor columns expected by
# melt_pairwise_to_query_records(...)

    def build_pairwise_sweep_table(sweep_df, cohort_name):
        rows = []
        n = len(sweep_df)

        for i in range(n):
            ri = sweep_df.iloc[i]

            for j in range(i + 1, n):
                rj = sweep_df.iloc[j]

                # keep the benchmark cross-donor, just like the main pipeline
                if ri["donor_name"] == rj["donor_name"]:
                    continue

                rows.append({
                    "cohort": cohort_name,
                    "id_i": int(ri["id"]),
                    "id_j": int(rj["id"]),
                    "label_i": ri["label"],
                    "label_j": rj["label"],
                    "donor_i": ri["donor_name"],
                    "donor_j": rj["donor_name"],

                    "full_sras": sras_np(ri["full_G"], rj["full_G"], eps=CFG["EPS_SPD"]),
                    "shape_sras": sras_np(ri["shape_G"], rj["shape_G"], eps=CFG["EPS_SPD"]),
                    "naive_full_sras": sras_np(ri["naive_full_G"], rj["naive_full_G"], eps=CFG["EPS_SPD"]),
                    "naive_shape_sras": sras_np(ri["naive_shape_G"], rj["naive_shape_G"], eps=CFG["EPS_SPD"]),
                })

        return pd.DataFrame(rows)
    def summarize_nmatch_sweep(source_df, cohort_name, values):
        out = []
        for n_match in values:
            tasks = [
                row for row in source_df.to_dict(orient="records")
                if int(row["n_cells"]) >= int(n_match)
            ]

            Parallel(n_jobs=CFG["N_JOBS_BUILD"], backend="loky", batch_size=1, verbose=0)(
                delayed(build_nmatch_sweep_cache)(row, cohort_name, n_match)
                for row in tasks
            )

            sweep_df = build_sweep_df(source_df, cohort_name, n_match)
            pair_df = build_pairwise_sweep_table(sweep_df, cohort_name)

            qr = melt_pairwise_to_query_records(pair_df)
            bench_df, _ = summarize_all_methods(qr, cohort_name)
            bench_df["n_match"] = int(n_match)
            bench_df["n_experiments"] = int(len(sweep_df))
            out.append(bench_df)

        return pd.concat(out, ignore_index=True) if len(out) else pd.DataFrame()

    area_nmatch_bench = summarize_nmatch_sweep(area_df, "area", AREA_SWEEP_VALUES)
    visp_nmatch_bench = summarize_nmatch_sweep(visp_df, "visp", VISP_SWEEP_VALUES)

    area_nmatch_bench.to_csv(TABLE_DIR / "area_nmatch_sweep_benchmark.csv", index=False)
    visp_nmatch_bench.to_csv(TABLE_DIR / "visp_nmatch_sweep_benchmark.csv", index=False)

    display(area_nmatch_bench)
    display(visp_nmatch_bench)
else:
    print("n_match sweep disabled.")

In [ ]:
# %%
# 27 — Final bundle export (truthful and complete)

def _df_records(df):
    if df is None or len(df) == 0:
        return []
    return df.to_dict(orient="records")

def _status_from_df(df, path=None):
    return {
        "available": bool(df is not None and len(df) > 0),
        "n_rows": int(len(df)) if df is not None else 0,
        "path": str(path) if path is not None else None,
    }

bundle = {
    "area_cohort": {
        "n_experiments": int(len(area_cache_df)),
        "n_match": int(N_MATCH_AREA),
        "count_control": _df_records(count_control_df[count_control_df["cohort"] == "area"]),
        "matching_benchmark": _df_records(area_match_df),
        "fisher_vs_naive": _df_records(area_fvn),
        "incremental_value": _df_records(incremental_df[incremental_df["cohort"] == "area"]),
        "allocation_summary": _df_records(area_alloc),
    },
    "visp_layer_cohort": {
        "n_experiments": int(len(visp_cache_df)),
        "n_match": int(N_MATCH_VISP),
        "count_control": _df_records(count_control_df[count_control_df["cohort"] == "visp"]),
        "matching_benchmark": _df_records(visp_match_df),
        "fisher_vs_naive": _df_records(visp_fvn),
        "incremental_value": _df_records(incremental_df[incremental_df["cohort"] == "visp"]),
    },
    "optional_modules": {
        "state": _status_from_df(
            state_summary if "state_summary" in globals() else None,
            TABLE_DIR / "state_summary_table.csv",
        ),
        "grouped_family_area": _status_from_df(
            area_family_match_df if "area_family_match_df" in globals() else None,
            TABLE_DIR / "area_family_matching.csv",
        ),
        "grouped_family_visp": _status_from_df(
            visp_family_match_df if "visp_family_match_df" in globals() else None,
            TABLE_DIR / "visp_family_matching.csv",
        ),
        "grouped_family_full_vs_shape_area": _status_from_df(
            area_full_vs_shape_df if "area_full_vs_shape_df" in globals() else None,
            TABLE_DIR / "area_family_full_vs_shape.csv",
        ),
        "grouped_family_full_vs_shape_visp": _status_from_df(
            visp_full_vs_shape_df if "visp_full_vs_shape_df" in globals() else None,
            TABLE_DIR / "visp_family_full_vs_shape.csv",
        ),
        "incremental_nuisance": _status_from_df(
            incremental_nuisance_df if "incremental_nuisance_df" in globals() else None,
            TABLE_DIR / "incremental_nuisance_models.csv",
        ),
        "nmatch_sweep_area": _status_from_df(
            area_nmatch_bench if "area_nmatch_bench" in globals() else None,
            TABLE_DIR / "area_nmatch_sweep_benchmark.csv",
        ),
        "nmatch_sweep_visp": _status_from_df(
            visp_nmatch_bench if "visp_nmatch_bench" in globals() else None,
            TABLE_DIR / "visp_nmatch_sweep_benchmark.csv",
        ),
        "coupling_area": _status_from_df(
            area_coupling_df if "area_coupling_df" in globals() else None,
            TABLE_DIR / "area_coupling_summary.csv",
        ),
        "coupling_visp": _status_from_df(
            visp_coupling_df if "visp_coupling_df" in globals() else None,
            TABLE_DIR / "visp_coupling_summary.csv",
        ),
    },
    "note": CFG["NOTEBOOK_VERSION"],
}

if "area_family_match_df" in globals():
    bundle["area_cohort"]["family_matching"] = _df_records(area_family_match_df)
if "visp_family_match_df" in globals():
    bundle["visp_layer_cohort"]["family_matching"] = _df_records(visp_family_match_df)

if "area_full_vs_shape_df" in globals():
    bundle["area_cohort"]["family_full_vs_shape"] = _df_records(area_full_vs_shape_df)
if "visp_full_vs_shape_df" in globals():
    bundle["visp_layer_cohort"]["family_full_vs_shape"] = _df_records(visp_full_vs_shape_df)

if "incremental_nuisance_df" in globals():
    bundle["area_cohort"]["incremental_nuisance"] = _df_records(
        incremental_nuisance_df[incremental_nuisance_df["cohort"] == "area"]
    )
    bundle["visp_layer_cohort"]["incremental_nuisance"] = _df_records(
        incremental_nuisance_df[incremental_nuisance_df["cohort"] == "visp"]
    )

if "area_coupling_df" in globals():
    bundle["area_cohort"]["coupling_summary"] = _df_records(area_coupling_df)
    bundle["area_cohort"]["coupling_by_label"] = _df_records(area_coupling_by_label_df)

if "visp_coupling_df" in globals():
    bundle["visp_layer_cohort"]["coupling_summary"] = _df_records(visp_coupling_df)
    bundle["visp_layer_cohort"]["coupling_by_label"] = _df_records(visp_coupling_by_label_df)

if "state_summary" in globals():
    bundle["state_module"] = {
        "n_rows": int(len(state_summary)),
        "summary": _df_records(state_summary),
    }

if "area_nmatch_bench" in globals():
    bundle["area_cohort"]["nmatch_sweep"] = _df_records(area_nmatch_bench)

if "visp_nmatch_bench" in globals():
    bundle["visp_layer_cohort"]["nmatch_sweep"] = _df_records(visp_nmatch_bench)

out_json = RESULTS_DIR / "biology_cleanbreak_bundle.json"
with open(out_json, "w") as f:
    json.dump(bundle, f, indent=2)

print("Saved:", out_json)
print(json.dumps(bundle["optional_modules"], indent=2))